In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")


Spark 4.1.0   catalog: lakehouse


DataFrame[]

In [2]:
BOOTSTRAP = "kafka:9092"
TOPIC = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [3]:
BRONZE_TABLE = "lakehouse.taxi.bronze_raw_events"
CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

# Create a raw Bronze table that mirrors Kafka payload + metadata as-is.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    key BINARY,
    value BINARY,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP,
    timestampType INT
) USING iceberg
""")

bronze_query = (
    raw_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable(BRONZE_TABLE)
)

print(f"Bronze stream started: {bronze_query.id}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Bronze stream started: 8cc6dbc7-0ccc-4c42-81e1-dbbdbf7fdc38
Checkpoint path: /home/jovyan/work/checkpoints/bronze_raw_events


In [4]:
zones = spark.read.parquet("../data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [5]:
raw = spark.read.table("lakehouse.taxi.bronze_raw_events")

raw.createOrReplaceTempView("bronze")

spark.sql("SELECT count(*) FROM bronze").show()
spark.sql("SELECT * FROM bronze LIMIT 3").show()

CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

+--------+
|count(1)|
+--------+
|     904|
+--------+

+----+--------------------+----------+---------+------+--------------------+-------------+
| key|               value|     topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------+---------+------+--------------------+-------------+
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|   228|2026-04-04 17:44:...|            0|
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|   229|2026-04-04 17:44:...|            0|
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|   230|2026-04-04 17:44:...|            0|
+----+--------------------+----------+---------+------+--------------------+-------------+



In [6]:
## SILVER

In [7]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType, IntegerType, TimestampType
)

In [8]:
## Define schema

In [9]:
TRIP_SCHEMA = StructType([
    StructField("VendorID",              LongType(),   True),
    StructField("tpep_pickup_datetime",  StringType(), True),   
    StructField("tpep_dropoff_datetime", StringType(), True),   
    StructField("passenger_count",       DoubleType(), True),   
    StructField("trip_distance",         DoubleType(), True),
    StructField("RatecodeID",            DoubleType(), True),
    StructField("store_and_fwd_flag",    StringType(), True),
    StructField("PULocationID",          LongType(),   True),
    StructField("DOLocationID",          LongType(),   True),
    StructField("payment_type",          LongType(),   True),
    StructField("fare_amount",           DoubleType(), True),
    StructField("extra",                 DoubleType(), True),
    StructField("mta_tax",               DoubleType(), True),
    StructField("tip_amount",            DoubleType(), True),
    StructField("tolls_amount",          DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount",          DoubleType(), True),
    StructField("congestion_surcharge",  DoubleType(), True),
    StructField("Airport_fee",           DoubleType(), True),
    StructField("cbd_congestion_fee",    DoubleType(), True),
])

In [10]:
## 2. Read bronze — batch

In [11]:
silver_batch = (
     spark.read
          .format("iceberg")                        
          .table("lakehouse.taxi.bronze_raw_events")
)


In [12]:

def parse(silver_read):
    # ---------------------------------------------------------------------------
    # 3. R1 — Parse JSON payload
    # ---------------------------------------------------------------------------
    parsed = (
        silver_batch
        .withColumn("value_str", F.col("value").cast("string"))
        .withColumn("trip",      F.from_json(F.col("value_str"), TRIP_SCHEMA))
        .select("trip.*",
                F.col("timestamp").alias("kafka_ingest_ts"))   # keep Kafka metadata
    )
    
    
    # ---------------------------------------------------------------------------
    # 4. R2 — Cast timestamp strings → TimestampType
    #    Input format: "2025-01-01T12:45:51" (ISO-8601 without timezone)
    # ---------------------------------------------------------------------------
    parsed = (
        parsed
        .withColumn("pickup_datetime",
                    F.to_timestamp("tpep_pickup_datetime",  "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("dropoff_datetime",
                    F.to_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd'T'HH:mm:ss"))
        .drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
    )
    return parsed


In [13]:
## Clean

In [14]:
def clean(parsed):
    cleaned = (
        parsed
    
        # R3 — passenger_count: float → int; must be 1–9
        .withColumn("passenger_count",
                    F.when(
                        F.col("passenger_count").cast(IntegerType()).between(1, 9),
                        F.col("passenger_count").cast(IntegerType())
                    ).otherwise(F.lit(None).cast(IntegerType())))
    
        # R4 — trip_distance must be positive
        .withColumn("trip_distance",
                    F.when(F.col("trip_distance") > 0, F.col("trip_distance"))
                     .otherwise(None))
    
        # R5 — RatecodeID valid domain: 1=Standard, 2=JFK, 3=Newark,
        #        4=Nassau/Westchester, 5=Negotiated, 6=Group ride
        .withColumn("RatecodeID",
                    F.when(F.col("RatecodeID").cast(IntegerType()).between(1, 6),
                           F.col("RatecodeID").cast(IntegerType()))
                     .otherwise(None))
    
        # R6 — payment_type valid domain: 1=Credit, 2=Cash, 3=No charge,
        #        4=Dispute, 5=Unknown, 6=Voided
        .withColumn("payment_type",
                    F.when(F.col("payment_type").between(1, 6), F.col("payment_type"))
                     .otherwise(None))
    
        # R7 — fare_amount cannot be negative
        .withColumn("fare_amount",
                    F.when(F.col("fare_amount") >= 0, F.col("fare_amount"))
                     .otherwise(None))
    
        # R8 — total_amount cannot be negative
        .withColumn("total_amount",
                    F.when(F.col("total_amount") >= 0, F.col("total_amount"))
                     .otherwise(None))
    
        # R9 — dropoff must be strictly after pickup
        .withColumn("is_valid_window",
                    F.col("dropoff_datetime") > F.col("pickup_datetime"))
        .filter(F.col("is_valid_window"))
        .drop("is_valid_window")
    
        # Derived: trip duration in minutes (convenient for analytics)
        .withColumn("trip_duration_min",
                    F.round(
                        (F.unix_timestamp("dropoff_datetime")
                         - F.unix_timestamp("pickup_datetime")) / 60.0,
                        2))
    
        # Partition column for Iceberg — date extracted from pickup
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )
    return cleaned


In [15]:
## Deduplicate

In [16]:
def dedup(cleaned):
    DEDUP_KEY = ["VendorID", "pickup_datetime", "dropoff_datetime",
                 "PULocationID", "DOLocationID"]
    return cleaned.dropDuplicates(DEDUP_KEY)


In [17]:
## Zone enrichment (pickup + dropoff)

In [18]:
def enrich(deduped):
    zones = spark.read.parquet("../data/taxi_zone_lookup.parquet").alias("z")
    
    enriched = (
        deduped.alias("t")
    
        # Pickup zone
        .join(
            zones.select(
                F.col("LocationID").alias("pu_loc_id"),
                F.col("Zone").alias("pickup_zone"),
                F.col("Borough").alias("pickup_borough"), 
                F.col("service_zone").alias("pickup_service_zone"),
            ),
            F.col("t.PULocationID") == F.col("pu_loc_id"),
            how="left"
        )
        .drop("pu_loc_id")
    
        # Dropoff zone
        .join(
            zones.select(
                F.col("LocationID").alias("do_loc_id"),
                F.col("Zone").alias("dropoff_zone"),
                F.col("Borough").alias("dropoff_borough"),
                F.col("service_zone").alias("dropoff_service_zone"),
            ),
            F.col("t.DOLocationID") == F.col("do_loc_id"),
            how="left"
        )
        .drop("do_loc_id")
    )
    return enriched
 


In [19]:
## Final column order

In [20]:
def order(enriched):
    silver = enriched.select(
        # identifiers
        "VendorID",
        # time
        "pickup_datetime", "dropoff_datetime", "trip_duration_min", "pickup_date",
        # geography
        "PULocationID", "pickup_zone",  "pickup_borough",
        "DOLocationID", "dropoff_zone", "dropoff_borough",
        # trip facts
        "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
        # fares
        "fare_amount", "extra", "mta_tax", "tip_amount",
        "tolls_amount", "improvement_surcharge",
        "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
        "total_amount", "payment_type", 
        # lineage
        "kafka_ingest_ts",
    )
    return silver

def transform(silver_stream):
    
    return order(enrich(dedup(clean(parse(silver_stream)))))


In [23]:
SILVER_CHECKPOINT = "/home/jovyan/work/checkpoints/bronze_raw_events"

silver_schema = transform(silver_batch).schema
table_name = "lakehouse.taxi.silver_trips"

if not spark.catalog.tableExists(table_name):
    print(f"Creating {table_name} for the first time...")
    
    spark.catalog.createTable(table_name, schema=silver_schema, source="iceberg")
    
    # Apply partitioning and optimizations via SQL
    spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD pickup_date")
    spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ('write.format.default'='parquet', 'write.parquet.compression-codec'='zstd')")
    print("Table initialized.")
else:
    print(f"{table_name} already exists. Skipping initialization.")

lakehouse.taxi.silver_trips already exists. Skipping initialization.


In [24]:
silver_batch_transformed = transform(silver_batch)

(
    silver_batch_transformed
    .writeTo("lakehouse.taxi.silver_trips")
    .overwritePartitions()
)

spark.read.table("lakehouse.taxi.silver_trips").count()


1536